# Lend a GPU to karaokie

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karaokie-app/colab/blob/main/karaokie_worker.ipynb)

Preparing a song means separating the voice from the music and placing every
word of the lyrics against it — a few minutes of work for a graphics card.
This notebook borrows the one Colab lends you and puts it on the queue at
[karaokie.app](https://karaokie.app), then hands the finished song back.

Nothing is installed on your own machine and there is no account to make with
karaokie. If you have never used Colab before, the whole of it is six steps.

---

### 1. Sign in to Google

Top right of this page, if it is asking. Colab is free and there is nothing to
download.

### 2. Ask for a graphics card

Open **Runtime** in the menu bar at the top of this page and choose **Change
runtime type**.

<img src="https://karaokie.app/guide/colab-menu.png" alt="The Runtime menu, with Change runtime type near the bottom" width="300">

Choose **T4 GPU** and press **Save**. It is usually already selected, because
this notebook asks for one.

<img src="https://karaokie.app/guide/colab-dialog.png?v=2" alt="The Change runtime type dialog, with T4 GPU selected" width="340">

This is the step worth not skipping. Without a card the work still runs, on the
processor, and a song takes something like ten times as long.

### 3. Press **Run all**

In the toolbar at the top of this page, next to *Code* and *Text*. (Or press
**Ctrl+F9**; **⌘+F9** on a Mac.) That runs both cells below, in order, which is
the entire job.

<img src="https://karaokie.app/guide/colab-toolbar.png" alt="Colab's toolbar, with Run all in the middle" width="560">

### 4. Say **Run anyway**

Colab will warn you that this notebook was not written by Google. Quite right —
it is written by karaokie, and every line it runs is on this page in front of
you. Press **Run anyway**.

### 5. Wait, and read the output

The first cell checks this machine and prints three lines: whether it got a
card, whether it can reach the internet, and whether YouTube will answer it.
If any of them says something is wrong, it says what to do about it.

The second cell downloads the worker and starts it. The first few minutes are
it fetching its own Python, ffmpeg and the audio models — several GB, once —
and after that it prints a line for every song: fetched, separated, aligned,
uploaded, finished.

### 6. Leave this tab open

That is the whole job. Songs will keep arriving for as long as it runs.

### While you are here: better separation

Cell 3 is optional and takes a minute. Without it the worker takes *every*
voice out of a song, harmonies included, and the backing track comes out
thinner than the record. With it, only the lead comes out and the backing
vocals stay -- which is what a karaoke track is supposed to sound like.

It is not part of the worker because it needs packages that will not build on
some of the machines people lend this project. On this one they install fine,
and once they are there the worker uses them by itself.
---

**To stop:** press the ■ beside the running cell, or **Runtime ▸ Interrupt
execution**. Nothing needs tidying up — a song caught half-done goes back on
the queue for somebody else a few minutes later.


In [ ]:
# 1 — is this runtime any use?
#
# Two things decide that, and Colab gives out both most of the time and
# neither some of the time. Better to find out here than three minutes into
# a song.
import subprocess

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

gpu = sh('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')
card = gpu.stdout.strip().splitlines()[0] if gpu.returncode == 0 and gpu.stdout.strip() else ''
if card:
    print('GPU       ', card)
else:
    print('GPU        none. Runtime > Change runtime type > T4 GPU, then run this again.')
    print('           It will still work on the processor, only far more slowly.')

print('YouTube    asking...')
sh('pip -q install -U yt-dlp')
# The same request the pipeline makes: music.youtube.com, over IPv4.
probe = sh("yt-dlp -4 --skip-download --no-warnings --print '%(title)s' "
           'https://music.youtube.com/watch?v=sQtnhwU2R9Y')
if probe.returncode == 0 and probe.stdout.strip():
    print('YouTube    ok --', probe.stdout.strip().splitlines()[-1])
else:
    said = (probe.stderr or '').strip().splitlines()
    print('YouTube    would not answer this machine:')
    print('          ', said[-1] if said else 'no answer at all')
    print()
    print('           Colab addresses are sometimes asked to prove they are not a')
    print('           bot, and a worker that cannot fetch a song is no use to the')
    print('           queue. Runtime > Disconnect and delete runtime, then connect')
    print('           again: the next machine has a different address and usually')
    print('           a different answer.')


In [ ]:
# 3 — better separation (optional, and worth it).
#
# The default separator takes every voice out of the song, harmonies
# included, and the backing track comes out thinner than the record. The
# karaoke RoFormer takes out only the lead and leaves the rest, which is what
# a commercial karaoke track sounds like.
#
# It is not a dependency of the worker, because it brings numba and llvmlite
# with it and those have no build for some of the machines people lend. On
# this one they install fine. Run this cell BEFORE cell 4 and the worker
# finds it by itself -- there is no second switch to set.
import os, subprocess, sys

HOME = os.path.expanduser('~/.karaokie-agent')
!curl -fsSL https://karaokie.app/install.sh | sh -s -- --no-run
# Builds the worker's private Python without taking a song, so there is
# something to install into.
!{HOME}/bin/karaokie-agent -setup
!{HOME}/runtime/bin/uv pip install --python {HOME}/runtime/venv/bin/python audio-separator audioread
print('\nready: the worker will use the karaoke separator')


In [ ]:
# 4 — run the worker.
#
# One binary, checked against the published checksum and started. The first
# few minutes are it fetching its own Python, ffmpeg and the audio models;
# after that it takes a song, prepares it, uploads it and asks for another.
#
# Press the stop button on this cell to stop it.
import os, subprocess

card = subprocess.run('nvidia-smi --query-gpu=name --format=csv,noheader',
                      shell=True, capture_output=True, text=True).stdout.strip()
# How this machine appears in the pool at karaokie.app/worker.
os.environ['KARAOKIE_NAME'] = f'colab {card.splitlines()[0]}' if card else 'colab'

!curl -fsSL https://karaokie.app/install.sh | sh


### While it runs

Each song is logged as it goes. A run of several hours fills that cell with a
lot of text — **Runtime ▸ Clear output** if the tab starts to feel heavy; it
does not interrupt the worker.

### When it stops

Colab hands the machine back after a while — sooner if the browser tab has been
idle, and after twelve hours at the outside. It may also refuse a graphics card
for a while if you have been using a lot of them. Nothing needs cleaning up: a
song that was being prepared when the machine went away loses its lease and is
offered to the next worker a few minutes later.

Colab is meant for interactive work, and a runtime that spends every hour of
every day at full tilt is the sort of thing its limits exist for. Run it while
you are around, stop it when you are not.

See who else is working, and every other way to run one, at
[karaokie.app/worker](https://karaokie.app/worker).
